# 03 Model Training

Train baseline and feature-rich models for points, rebounds, and assists.

This notebook now includes:
- `LinearRegression`
- `RandomForestRegressor`
- `XGBRegressor`

It also saves prediction-versus-actual evaluation datasets for downstream analysis.

## Imports And Load

In [1]:
from pathlib import Path
import time

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

ROOT = Path.cwd()
FINAL_DIR = ROOT / 'data' / 'final'
RESULTS_DIR = ROOT / 'outputs' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SEASONS = ['2023-24']
EVAL_SEASON = '2024-25'
WANDB_ENABLED = False

df = pd.read_parquet(FINAL_DIR / 'ml_ready_player_games.parquet')
train_df = df[df['season'].isin(TRAIN_SEASONS)].copy()
eval_df = df[df['season'] == EVAL_SEASON].copy()
train_df.shape, eval_df.shape

((26401, 70), (26306, 70))

## Define Targets And Feature Sets

In [2]:
TARGETS = {
    'target_pts': {'BaselineA': ['season_avg_pts'], 'BaselineB': ['l5_avg_pts']},
    'target_reb': {'BaselineA': ['season_avg_reb'], 'BaselineB': ['l5_avg_reb']},
    'target_ast': {'BaselineA': ['season_avg_ast'], 'BaselineB': ['l5_avg_ast']},
}

def get_feature_sets(target):
    v0 = [
        'season_avg_pts', 'season_avg_reb', 'season_avg_ast', 'season_avg_min', 'season_avg_fga', 'season_avg_fg3a', 'season_avg_fta',
        'l5_avg_pts', 'l5_avg_reb', 'l5_avg_ast', 'l5_avg_min', 'l5_avg_fga', 'l5_avg_fg3a', 'l5_avg_fta',
        'l10_avg_pts', 'l10_avg_reb', 'l10_avg_ast', 'l10_avg_min', 'l10_avg_fga', 'l10_avg_fg3a', 'l10_avg_fta',
        'opponent_team', 'home_away'
    ]
    availability = [
        'teammates_out_count', 'missing_pts_l5', 'missing_reb_l5', 'missing_ast_l5', 'missing_min_l5',
        'missing_pts_l10', 'missing_reb_l10', 'missing_ast_l10', 'missing_min_l10'
    ]
    usage_context = ['season_avg_usg', 'l5_avg_usg', 'l10_avg_usg', 'missing_usg_l5', 'missing_usg_l10', 'opponent_pace', 'opponent_def_rating']
    return {
        'BaselineA': TARGETS[target]['BaselineA'],
        'BaselineB': TARGETS[target]['BaselineB'],
        'V0': v0,
        'V1': v0 + availability,
        'V2': v0 + availability + usage_context,
    }

get_feature_sets('target_pts').keys()

dict_keys(['BaselineA', 'BaselineB', 'V0', 'V1', 'V2'])

## Modeling Helpers

In [3]:
def make_pipeline(feature_cols, model_family):
    categorical = [c for c in feature_cols if c in ['opponent_team', 'home_away']]
    numeric = [c for c in feature_cols if c not in categorical]
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
    ])
    if model_family == 'LinearRegression':
        model = LinearRegression()
    elif model_family == 'RandomForestRegressor':
        model = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1)
    elif model_family == 'XGBRegressor':
        model = XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='reg:squarederror',
            random_state=42,
            n_jobs=4,
        )
    else:
        raise ValueError(model_family)
    return Pipeline([('preprocessor', preprocessor), ('model', model)])

## Train And Evaluate

In [4]:
wandb_run = None
if WANDB_ENABLED:
    import wandb
    wandb_run = wandb.init(project='nba-statline-forecasting', job_type='training', reinit=True)

runs = []
eval_prediction_frames = []
model_families = ['LinearRegression', 'RandomForestRegressor', 'XGBRegressor']

for target in TARGETS:
    print(f'Running target={target}')
    for model_version, feature_cols in get_feature_sets(target).items():
        print(f'  model_version={model_version}')
        for model_family in model_families:
            print(f'    model_family={model_family} starting')
            start = time.time()
            pipeline = make_pipeline(feature_cols, model_family)
            pipeline.fit(train_df[feature_cols], train_df[target])
            preds = pipeline.predict(eval_df[feature_cols])
            mae = mean_absolute_error(eval_df[target], preds)
            rmse = mean_squared_error(eval_df[target], preds) ** 0.5
            run = {
                'run_id': f'{target}_{model_version}_{model_family}',
                'target': target,
                'model_version': model_version,
                'model_family': model_family,
                'feature_group': ','.join(feature_cols),
                'train_seasons': ','.join(TRAIN_SEASONS),
                'eval_season': EVAL_SEASON,
                'mae': mae,
                'rmse': rmse,
                'runtime_seconds': time.time() - start,
                'notes': '',
            }
            runs.append(run)

            pred_df = eval_df[['player_id', 'player_name', 'game_id', 'game_date', 'season', 'team_abbr', 'opponent_team', 'home_away', target]].copy()
            pred_df = pred_df.rename(columns={target: 'actual'})
            pred_df['prediction'] = preds
            pred_df['error'] = pred_df['prediction'] - pred_df['actual']
            pred_df['abs_error'] = pred_df['error'].abs()
            pred_df['squared_error'] = pred_df['error'] ** 2
            pred_df['run_id'] = run['run_id']
            pred_df['target'] = target
            pred_df['model_version'] = model_version
            pred_df['model_family'] = model_family
            pred_df.to_parquet(RESULTS_DIR / f'eval_predictions_{target}_{model_version}_{model_family}.parquet', index=False)
            eval_prediction_frames.append(pred_df)

            if wandb_run is not None:
                wandb_run.log(run)
            print(f'    model_family={model_family} done mae={mae:.4f} rmse={rmse:.4f}')

if wandb_run is not None:
    wandb_run.finish()

results_df = pd.DataFrame(runs).sort_values(['target', 'model_version', 'model_family']).reset_index(drop=True)
results_df.to_csv(RESULTS_DIR / 'model_results.csv', index=False)

eval_predictions_long = pd.concat(eval_prediction_frames, ignore_index=True)
eval_predictions_long.to_parquet(RESULTS_DIR / 'eval_predictions_long.parquet', index=False)
eval_predictions_long.to_csv(RESULTS_DIR / 'eval_predictions_long.csv', index=False)

best_runs = results_df.loc[results_df.groupby('target')['mae'].idxmin()].copy()
best_eval_predictions = eval_predictions_long.merge(best_runs[['run_id']], on='run_id', how='inner')
best_eval_predictions.to_parquet(RESULTS_DIR / 'best_eval_predictions.parquet', index=False)
best_eval_predictions.to_csv(RESULTS_DIR / 'best_eval_predictions.csv', index=False)

results_df

Running target=target_pts
  model_version=BaselineA
    model_family=LinearRegression starting
    model_family=LinearRegression done mae=4.7489 rmse=6.1950
    model_family=RandomForestRegressor starting
    model_family=RandomForestRegressor done mae=5.1078 rmse=6.6922
    model_family=XGBRegressor starting
    model_family=XGBRegressor done mae=4.7596 rmse=6.2095
  model_version=BaselineB
    model_family=LinearRegression starting
    model_family=LinearRegression done mae=4.8438 rmse=6.3195
    model_family=RandomForestRegressor starting
    model_family=RandomForestRegressor done mae=4.8685 rmse=6.3530
    model_family=XGBRegressor starting
    model_family=XGBRegressor done mae=4.8548 rmse=6.3324
  model_version=V0
    model_family=LinearRegression starting
    model_family=LinearRegression done mae=4.6542 rmse=6.0838
    model_family=RandomForestRegressor starting
    model_family=RandomForestRegressor done mae=4.7677 rmse=6.2069
    model_family=XGBRegressor starting
    model_

,run_id,target,model_version,model_family,feature_group,train_seasons,eval_season,mae,rmse,runtime_seconds,notes
0,target_ast_BaselineA_LinearRegression,target_ast,BaselineA,LinearRegression,season_avg_ast,2023-24,2024-25,1.372025,1.863006,0.013806,
1,target_ast_BaselineA_RandomForestRegressor,target_ast,BaselineA,RandomForestRegressor,season_avg_ast,2023-24,2024-25,1.458695,1.998495,0.675006,
2,target_ast_BaselineA_XGBRegressor,target_ast,BaselineA,XGBRegressor,season_avg_ast,2023-24,2024-25,1.378558,1.868841,0.319800,
3,target_ast_BaselineB_LinearRegression,target_ast,BaselineB,LinearRegression,l5_avg_ast,2023-24,2024-25,1.406466,1.907371,0.009473,
4,target_ast_BaselineB_RandomForestRegressor,target_ast,BaselineB,RandomForestRegressor,l5_avg_ast,2023-24,2024-25,1.409654,1.910595,0.368701,
5,target_ast_BaselineB_XGBRegressor,target_ast,BaselineB,XGBRegressor,l5_avg_ast,2023-24,2024-25,1.409934,1.912698,0.294302,
6,target_ast_V0_LinearRegression,target_ast,V0,LinearRegression,"season_avg_pts,season_avg_reb,season_avg_ast,s...",2023-24,2024-25,1.349590,1.838608,0.132482,
7,target_ast_V0_RandomForestRegressor,target_ast,V0,RandomForestRegressor,"season_avg_pts,season_avg_reb,season_avg_ast,s...",2023-24,2024-25,1.390801,1.879293,11.637265,
8,target_ast_V0_XGBRegressor,target_ast,V0,XGBRegressor,"season_avg_pts,season_avg_reb,season_avg_ast,s...",2023-24,2024-25,1.364798,1.861474,0.980057,
9,target_ast_V1_LinearRegression,target_ast,V1,LinearRegression,"season_avg_pts,season_avg_reb,season_avg_ast,s...",2023-24,2024-25,1.352002,1.830642,0.163743,


## Compare Results

In [5]:
results_df.pivot(index=['target', 'model_version'], columns='model_family', values='mae').round(3)

model_family              LinearRegression  RandomForestRegressor  \
target     model_version                                            
target_ast BaselineA                 1.372                  1.459   
           BaselineB                 1.406                  1.410   
           V0                        1.350                  1.391   
           V1                        1.352                  1.384   
           V2                        1.348                  1.378   
target_pts BaselineA                 4.749                  5.108   
           BaselineB                 4.844                  4.869   
           V0                        4.654                  4.768   
           V1                        4.640                  4.716   
           V2                        4.614                  4.671   
target_reb BaselineA                 1.969                  2.099   
           BaselineB                 2.008                  2.010   
           V0                        1.943                  1.983   
           V1                        1.943                  1.966   
           V2                        1.946                  1.966   

model_family              XGBRegressor  
target     model_version                
target_ast BaselineA             1.379  
           BaselineB             1.410  
           V0                    1.365  
           V1                    1.357  
           V2                    1.353  
target_pts BaselineA             4.760  
           BaselineB             4.855  
           V0                    4.689  
           V1                    4.679  
           V2                    4.611  
target_reb BaselineA             1.974  
           BaselineB             2.011  
           V0                    1.958  
           V1                    1.946  
           V2                    1.950

## Preview Prediction-Vs-Actual Output

In [6]:
eval_predictions_long.head()

,player_id,player_name,game_id,game_date,season,team_abbr,opponent_team,home_away,actual,prediction,error,abs_error,squared_error,run_id,target,model_version,model_family
0,2544,LeBron James,0022400062,2024-10-22,2024-25,LAL,MIN,home,16,9.013758,-6.986242,6.986242,48.807581,target_pts_BaselineA_LinearRegression,target_pts,BaselineA,LinearRegression
1,2544,LeBron James,0022400085,2024-10-25,2024-25,LAL,PHX,home,21,15.980907,-5.019093,5.019093,25.191295,target_pts_BaselineA_LinearRegression,target_pts,BaselineA,LinearRegression
2,2544,LeBron James,0022400096,2024-10-26,2024-25,LAL,SAC,home,32,18.334674,-13.665326,13.665326,186.741146,target_pts_BaselineA_LinearRegression,target_pts,BaselineA,LinearRegression
3,2544,LeBron James,0022400111,2024-10-28,2024-25,LAL,PHX,away,11,22.571454,11.571454,11.571454,133.898537,target_pts_BaselineA_LinearRegression,target_pts,BaselineA,LinearRegression
4,2544,LeBron James,0022400118,2024-10-30,2024-25,LAL,CLE,away,26,19.746934,-6.253066,6.253066,39.100840,target_pts_BaselineA_LinearRegression,target_pts,BaselineA,LinearRegression
